# Fase 6: Ingeniería de características (Feature Preprocessing)
En este notebook aplicamos las conclusiones obtenidas durante la fase de Exploración de Datos (EDA) sobre el dataset de reservas de **Oasis Spa Sevilla**. El objetivo es transformar el dataset bruto en una matriz de características puramente numérica y óptima para el entrenamiento de nuestros modelos de Machine Learning.

### Objetivos del preprocesamiento:
1. **Modelar el comportamiento temporal:** Agrupar variables de calendario para evitar sobreajuste y capturar patrones reales de demanda.
2. **Aplicar Reglas de Negocio:** Excluir datos ruidosos o no operativos (como los días de cierre del spa).
3. **Eliminar Redundancias:** Evitar la multicolinealidad eliminando variables correlacionadas o duplicadas.
4. **Automatización:** Validar la función centralizada `build_features` que se utilizará tanto en entrenamiento como en producción.

In [5]:
import sys
import os
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from src.utils.preprocessing import build_features


In [6]:
#Carga del dataset
ruta_train = '../data_sample/processed/train_features.csv' 
ruta_test = '../data_sample/processed/test_features.csv'
train_raw = pd.read_csv(ruta_train)
test_raw = pd.read_csv(ruta_test)

print("Datasets brutos cargados")
print(f"[Train, Filas iniciales: {train_raw.shape[0]} | Columnas: {len(train_raw.columns)}")
print(f"[Test,  Filas iniciales: {test_raw.shape[0]} | Columnas: {len(test_raw.columns)}")
print(f"\nColumnas iniciales en los datos: {train_raw.columns.tolist()}")

Datasets brutos cargados
[Train, Filas iniciales: 1252 | Columnas: 18
[Test,  Filas iniciales: 314 | Columnas: 18

Columnas iniciales en los datos: ['fecha_cita', 'tramo', 'n_citas', 'dia_semana', 'nombre_dia', 'es_finde', 'mes', 'anio', 'semana_iso', 'trimestre', 'dias_desde_inicio', 'grupo_dia', 'temporada', 'tramo_tarde', 'es_festivo', 'es_vispera_festivo', 'es_fecha_comercial', 'es_cierre']


## Justificación de las transformaciones aplicadas

Para asegurar un modelo robusto y evitar que aprenda patrones sesgados, hemos aplicado las siguientes transformaciones matemáticas y de negocio en `src/preprocessing.py`:

### 1. Reglas de negocio y Llimpieza de filas
* **`es_cierre` (Exclusión de filas):** Los días en los que el Oasis Spa Sevilla permanece cerrado no aportan información sobre el comportamiento natural de la demanda (las citas son obligatoriamente 0). Entrenar con estos días sesgaría al modelo a la baja. Los excluimos del set de entrenamiento. En producción, una regla simple devolverá `0` citas en días de cierre sin llegar a consultar al modelo.

### 2. Ingeniería y reducción de redundancias (Columnas)
* **`tramo_tarde` en favor de `tramo`:** Transformamos la variable categórica `tramo` ("mañana"/"tarde") en una variable binaria donde `1` representa la tarde y `0` la mañana.
* **`grupo_dia` en favor de `nombre_dia` y `es_finde`:** El análisis del EDA demostró que la demanda se comporta en tres bloques claros (Lunes-Jueves, Viernes, y Fin de semana). Agruparlos reduce la cardinalidad de la variable y evita que el modelo sobreajuste para días específicos de la semana.
* **Señales débiles (`es_festivo` y `es_vispera_festivo`):** Aunque mostraron una correlación débil en el EDA, se mantienen inicialmente en el modelo para evaluar su impacto real en el entrenamiento. Serán las primeras candidatas a eliminar si decidimos simplificar el modelo (Principio de parsimonia).

### 3. Eliminación de variables temporales lineales
* Se eliminan las columnas originales de fecha (`fecha_cita`, `mes`, `anio`, `dia_semana`, `semana_iso`) para evitar que el modelo asuma relaciones lineales incorrectas con variables cíclicas.
* Se genera la variable **`dias_desde_inicio`** como nuestra variable de tendencia global para capturar el crecimiento del negocio en el tiempo de forma lineal.

In [7]:
# 1.Transformar los datos de train y test aplicando build_features
X_train, y_train = build_features(train_raw)
X_test, y_test = build_features(test_raw)

print("Control de calidad de los datasets procesados")
print(f"Train, Filas resultantes (sin cierres): {X_train.shape[0]} | Columnas: {X_train.shape[1]}")
print(f"Test,  Filas resultantes (sin cierres): {X_test.shape[0]} | Columnas: {X_test.shape[1]}")

print(f"\n¿Quedan valores nulos en Train?: {X_train.isnull().sum().sum()}")
print(f"¿Quedan valores nulos en Test?:  {X_test.isnull().sum().sum()}")

#Comprobamos que todas las columnas son puramente numéricas en ambos
train_numerico = all(X_train.dtypes != 'object')
test_numerico = all(X_test.dtypes != 'object')

print(f"\n¿Son todas las columnas numéricas en Train?: {train_numerico}")
print(f"¿Son todas las columnas numéricas en Test?:  {test_numerico}")

#Comprobación de consistencia entre columnas
mismas_columnas = list(X_train.columns) == list(X_test.columns)
print(f"¿Tienen Train y Test exactamente las mismas columnas?: {mismas_columnas}")

Control de calidad de los datasets procesados
Train, Filas resultantes (sin cierres): 1240 | Columnas: 15
Test,  Filas resultantes (sin cierres): 314 | Columnas: 14

¿Quedan valores nulos en Train?: 0
¿Quedan valores nulos en Test?:  0

¿Son todas las columnas numéricas en Train?: True
¿Son todas las columnas numéricas en Test?:  True
¿Tienen Train y Test exactamente las mismas columnas?: False


## Control de calidad y tratamiento de datos (Missing Values & Drops)

Siguiendo las buenas prácticas metodológicas, evaluamos la necesidad de aplicar técnicas de imputación y selección automática de variables sobre nuestro conjunto de datos:

### 1. Gestión de valores nulos (Imputación)
En problemas de previsión de demanda con componente temporal, la imputación mediante medidas de tendencia central global (como la mediana o la media) puede distorsionar la estacionalidad diaria o semanal del negocio. 
* **Situación en Oasis Spa Sevilla:** Tras la fase de EDA, se ha verificado que el dataset no presenta registros nulos en las variables clave de calendario, tramos horarios o histórico de citas. 
* **Decisión:** Omitimos de forma intencionada el uso de herramientas como `SimpleImputer` para preservar la pureza temporal de la serie y evitar sesgar las predicciones futuras.

### 2. Selección de características y descarte de columnas (Feature Selection)
En lugar de aplicar un umbral genérico de descarte por porcentaje de nulos o varianza constante:
* Hemos realizado un filtrado guiado por **reglas de negocio** (exclusión de filas con `es_cierre == 1`).
* Descartamos variables redundantes analizadas en el EDA (`es_finde`, `tramo`, `nombre_dia`) directamente dentro de nuestro pipeline de preprocesamiento, sustituyéndolas por sus versiones optimizadas de menor cardinalidad (`grupo_dia`, `tramo_tarde`).

## Codificación de variables y escalado de características

### 1. Automatización y consistencia del Encoding (train & test)
Para evitar el riesgo de *Data Leakage* (fuga de datos) y garantizar la consistencia en el modelado, la codificación de variables categóricas (como los tramos horarios y las agrupaciones de días de la semana) se realiza de manera controlada y hermética dentro de nuestra función modular `build_features()`. 

Al aplicar esta misma función tanto a `train_raw` como a `test_raw`, garantizamos de forma idéntica:
* La eliminación de columnas excluidas o redundantes analizadas en el EDA (como `es_finde` o `nombre_dia`).
* Que el conjunto de entrenamiento y el de prueba adopten exactamente las mismas columnas numéricas y la misma estructura matemática final.

### 2. Prevención de Data Leakage en el escalado (feature scaling)
Dado que los algoritmos basados en árboles de decisión (como *Random Forest* o *XGBoost*) son invariantes a la escala, el escalado no es estrictamente necesario para los modelos de ensamble. Sin embargo, para poder compararlos de manera justa con modelos lineales (nuestro *Baseline*), aplicaremos una estandarización mediante `StandardScaler` a nuestra variable de tendencia continua (`dias_desde_inicio`).

Para simular un escenario real de producción y evaluar correctamente el modelo, seguimos una metodología estricta:
1. **Ajuste y transformación (`fit_transform`):** El objeto `StandardScaler` calcula la media ($\mu$) y la desviación estándar ($\sigma$) basándose **únicamente** en el conjunto de entrenamiento (`X_train`).
2. **Transformación pura (`transform`):** Aplicamos estos parámetros previamente calculados sobre el conjunto de test (`X_test`) sin recalcularlos. Esto impide que la información del conjunto de prueba "contamine" el entrenamiento del modelo. El escalador se exporta mediante `joblib` para poder usar exactamente la misma escala con datos nuevos en producción.

In [8]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib


print("Aplicar build_features a train y test")
#Aplicar la función de preprocesamiento del script .py a ambos conjuntos por separado
X_train, y_train = build_features(train_raw)
X_test, y_test = build_features(test_raw)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de X_test:  {X_test.shape}")

#Aplicar el escalado de forma segura
cols_a_escalar = ['dias_desde_inicio']

if all(col in X_train.columns for col in cols_a_escalar) and all(col in X_test.columns for col in cols_a_escalar):
    scaler = StandardScaler()
    
    #Ajustar y transformar únicamente con el conjunto de train
    X_train[cols_a_escalar] = scaler.fit_transform(X_train[cols_a_escalar])
    
    #Transformar el conjunto de test usando los parámetros de train
    X_test[cols_a_escalar] = scaler.transform(X_test[cols_a_escalar])
    
    #Creamos la carpeta de destino para el modelo si no existe
    os.makedirs('../models', exist_ok=True)
    
    #Guardamos el escalador ajustado para el entorno de producción
    joblib.dump(scaler, '../models/scaler.joblib')
    
    print("\nEscalado completado")
    print(f"Se ha guardado el escalador entrenado en: 'src/models/scaler.joblib'")
else:
    print("\n[Aviso] No se encontró la columna 'dias_desde_inicio' en alguno de los conjuntos.")

#Mostramos el resultado final de ambos conjuntos listos para modelar
print("\nVista previa del conjunto de entrenamiento procesado (X_train)")
display(X_train.head(2))

print("\nVista previa del conjunto de prueba procesado (X_test)")
display(X_test.head(2))

Aplicar build_features a train y test
Dimensiones de X_train: (1240, 15)
Dimensiones de X_test:  (314, 14)

Escalado completado
Se ha guardado el escalador entrenado en: 'src/models/scaler.joblib'

Vista previa del conjunto de entrenamiento procesado (X_train)


,dia_semana,mes,trimestre,dias_desde_inicio,tramo_tarde,es_festivo,es_vispera_festivo,es_fecha_comercial,grupo_dia_entre_semana,grupo_dia_fin_de_semana,grupo_dia_viernes,temporada_invierno,temporada_otoño,temporada_primavera,temporada_verano
0,3,5,2,-1.726688,0,0,0,0,1,0,0,0,0,1,0
1,3,5,2,-1.726688,1,0,0,0,1,0,0,0,0,1,0



Vista previa del conjunto de prueba procesado (X_test)


,dia_semana,mes,trimestre,dias_desde_inicio,tramo_tarde,es_festivo,es_vispera_festivo,es_fecha_comercial,grupo_dia_entre_semana,grupo_dia_fin_de_semana,grupo_dia_viernes,temporada_invierno,temporada_primavera,temporada_verano
0,6,1,1,1.743641,0,0,0,0,0,1,0,1,0,0
1,6,1,1,1.743641,1,0,0,0,0,1,0,1,0,0
